In [ ]:
import torch
import torch.nn as nn

torch.set_default_device('cpu')
######################updated after using git 
class Discriminator(nn.Module):
    def __init__(self, seq_length=3000, dropout=0.3):
        super(Discriminator, self).__init__()

        self.seq_length = seq_length

        # Conv Block 1: (batch, 3000, 1) -> (batch, 1500, 64)
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=64, kernel_size=25, stride=2, padding=12),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Conv Block 2: (batch, 1500, 64) -> (batch, 750, 128)
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Conv Block 3: (batch, 750, 128) -> (batch, 375, 256)
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Conv Block 4: (batch, 375, 256) -> (batch, 187, 512)
        self.conv4 = nn.Sequential(
            nn.Conv1d(256, 512, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Conv Block 5: (batch, 187, 512) -> (batch, 93, 512)
        self.conv5 = nn.Sequential(
            nn.Conv1d(512, 512, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Calculate flattened size after convolutions
        # 3000 -> 1500 -> 750 -> 375 -> 188 -> 94
        self.flatten_size = 94 * 512

        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Ensure input is (batch, channels, length)
        if x.shape[-1] == 1:
            x = x.permute(0, 2, 1)  # (batch, seq_length, 1) -> (batch, 1, seq_length)

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.fc(x)

        return x


class ConditionalDiscriminator(nn.Module):
    def __init__(self, seq_length=3000, num_classes=3, embedding_dim=50, dropout=0.3):
        super(ConditionalDiscriminator, self).__init__()

        self.seq_length = seq_length
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim

        # Label embedding
        self.label_embedding = nn.Embedding(num_classes, embedding_dim)

        # Project embedding to match sequence length
        self.label_proj = nn.Linear(embedding_dim, seq_length)

        # Conv Block 1: (batch, 3000, 2) -> (batch, 1500, 64)
        # Note: 2 channels because we concatenate signal + embedded label
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=64, kernel_size=25, stride=2, padding=12),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Conv Block 2: (batch, 1500, 64) -> (batch, 750, 128)
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Conv Block 3: (batch, 750, 128) -> (batch, 375, 256)
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Conv Block 4: (batch, 375, 256) -> (batch, 187, 512)
        self.conv4 = nn.Sequential(
            nn.Conv1d(256, 512, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Conv Block 5: (batch, 187, 512) -> (batch, 93, 512)
        self.conv5 = nn.Sequential(
            nn.Conv1d(512, 512, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        self.flatten_size = 94 * 512

        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        # Ensure input is (batch, 1, seq_length)
        if x.shape[-1] == 1:
            x = x.permute(0, 2, 1)

        # Embed labels and project to sequence length
        label_embed = self.label_embedding(labels)  # (batch, embedding_dim)
        label_proj = self.label_proj(label_embed)   # (batch, seq_length)
        label_proj = label_proj.unsqueeze(1)        # (batch, 1, seq_length)

        # Concatenate signal with label
        x = torch.cat([x, label_proj], dim=1)       # (batch, 2, seq_length)

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.fc(x)

        return x


def test_discriminator():
    # Test standard discriminator
    disc = Discriminator(seq_length=3000, dropout=0.3)

    # Create dummy batch
    batch_size = 100
    seq_length = 3000
    dummy_input = torch.randn(batch_size, seq_length, 1)

    output = disc(dummy_input)

    print(f"Input shape: {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Output range: [{output.min().item():.4f}, {output.max().item():.4f}]")

    # Count parameters
    total_params = sum(p.numel() for p in disc.parameters())
    trainable_params = sum(p.numel() for p in disc.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    print("\n" + "="*60)
    print("Testing Conditional Discriminator...")

    # Test conditional discriminator
    cond_disc = ConditionalDiscriminator(seq_length=3000, num_classes=3, dropout=0.3)

    dummy_labels = torch.randint(0, 3, (batch_size,))
    output = cond_disc(dummy_input, dummy_labels)

    print(f"Input shape: {dummy_input.shape}")
    print(f"Labels shape: {dummy_labels.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Output range: [{output.min().item():.4f}, {output.max().item():.4f}]")

    total_params = sum(p.numel() for p in cond_disc.parameters())
    trainable_params = sum(p.numel() for p in cond_disc.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")


if __name__ == '__main__':
    test_discriminator()

Testing Discriminator...
Input shape: torch.Size([100, 3000, 1])
Output shape: torch.Size([100, 1])
Output range: [0.3376, 0.8709]
Total parameters: 10,908,417
Trainable parameters: 10,908,417

Testing Conditional Discriminator...
Input shape: torch.Size([100, 3000, 1])
Labels shape: torch.Size([100])
Output shape: torch.Size([100, 1])
Output range: [0.2292, 0.7505]
Total parameters: 11,063,167
Trainable parameters: 11,063,167

✓ All tests passed!
